## MINI GPT MODEL BASED ON TRANSFORMER ARCHITECTURE ##

#### GPT TRANSFORMER ARCHITECTURE  ####


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## NOTE below we using many parameters so there meaning are 
1) B ==> batch size
2) T = block_size = Context window ==> number of tokens
3) C ==> embedding dimension


In [ ]:
#Q=X*wq
#K=X*Wk
#V = X*Wv

## so below let's create Wq,Wk,Wv learnable matrix

## Why head_size?

## Your model has:
## embedding_dim = 32
## n_heads = 2
## head_size = embedding_dim // num_heads = 16
## Each Head calculate attention score separately



In [2]:
class SelfAttentionzHead(nn.Module):
    def __init__(self,embedding_dim,block_size,head_size):
        super().__init__()
        self.query = nn.Linear(embedding_dim,head_size,bias=False)  ## here we create Wq matrix of (32,16) 
        self.key = nn.Linear(embedding_dim,head_size,bias=False)  ## here we create Wk matrix
        self.value = nn.Linear(embedding_dim,head_size,bias=False) ## here we create Wq matrix
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))  ## here we create lower trangular matrix for Causal masking and store it in register_buffer

    # in below forward pass function we calculate attention score 
    #Attention(Q,K,V)=softmax(Q*KT/squareroot(d)​)V
    def forward(self,x):
        B,T,C = x.shape  ## here we extract  batch_size,block_size(context_window) and embedding_size from input
        q = self.query(x)   ## here we create query and key representation from input ==> Q = x*Wq
        k = self.query(x)   ## similarly for key ==> K = x*WK
        wei = q @ k.transpose(-2,-1) ## here we calculate dot product between each query matrix of token with other keys matrix (Attrntion score) 
        # Note symbol @ is used for dot product calculation
        wei = wei/(C ** 0.5)  ## here we scale dot product operation in order to prevent exponent softmax problem by diving score square root of embedding dimention
        wei = F.softmax(wei,dim=-1) ## calculate Softmax
        v = self.value(x) ## calculate value matrix 
        out = wei @ v ## finally we do dot product between scale attention score with value
        return out

## MULTIHEAD Attention
Multi-head attention in transformer architecture is necessary 
because a single attention head can only focus on one 
relationship or perspective at a time

--> Why Multiple Heads Are Needed
1) Multiple Perspectives: Different heads analyze the input 
   from various angles simultaneously.
2) Diverse Relationships: One head might track grammar or 
   subject-verb links, while another tracks long-range context 
   or semantic meaning.
3) Resolving Ambiguity: Processing multiple interpretations at 
   once helps the model handle complex, ambiguous language.
4) Computational Efficiency: Splitting the embedding dimension across 
   multiple smaller heads keeps the total computation cost similar to a single large attention pass

In [3]:
## let's build MultiHead Attention
class MultiHeadAttention(nn.Module):
    def ___init__(self,embedding_dim,block_size,num_heads):
        super().__init__()
        head_size = embedding_dim // num_heads   ## here we mention size of head (numbers of input head can process) in our case 32/2 = 16
        self.heads = ([SelfAttentionzHead(embedding_dim,block_size,head_size) for _ in range(num_heads)]) ## This creates multiple SelfAttentionHead objects.
        # Each head receives the same embedding_dim, block_size, and head_size. 
        # After all heads produce their outputs, they are concatenated:
        self.proj = nn.Linear(num_heads * head_size,embedding_dim) # self.proj then learns how to mix the information from all heads and converts it back to the original embedding_dim size

    # In forward(), [h(x) for h in self.heads] sends inputs through every attention head.
    def forward(self,x):
        out = torch.concat([h(x) for h in self.heads],dim=-1) # combines all head outputs back into one larger representation
        return self.proj(out)

## FNN (Feed Forward network)
FNN stands for Feed-Forward Network (also called FFN), a component inside each Transformer block. 
It sits after the self-attention layer and processes each token's representation independently 
(position-wise), applying two linear transformations with a non-linear activation (like ReLU or GELU) in between: 
essentially Linear → Activation → Linear. 
Its role is to add non-linearity and increase the model's capacity to transform features, 
since attention alone only mixes information across tokens but doesn't add much representational depth. 
Typically the hidden layer inside the FFN is much larger (often 4x) than the model's embedding dimension,
expanding then compressing the representation. 
Each Transformer layer has its own separate FFN with its own weights, and it's usually followed 
by a residual connection and layer normalization.


In [4]:
## let build FNN in our code
class FNN(nn.Module):
    def __init__(self,n_embd):
        super().__init__()
        self.net = nn.Sequential(    ## here we use Sequential which connect 3 important operation Expansion ---> Add Non Linearity ---> Compresion 
            nn.Linear(n_embd,4*n_embd),   ## nn.Linear() expand each token from 32 to 128 diemension. purpose of this expansion is to give the network large feature space to learn complex pattern
            F.relu(),  ## here we add non linearity into model which allow model to learn more than just linear relationship
            nn.Linear(4*n_embd,n_embd)  ## at the last here we compress 128 diamension back to 32 cause we have to maintain the dimensionality across the attention layer.
        )
    def forward(self,x):  ## here we simply pass the input through above feed forward network
        out = self.net(x)
        return out

#### TRANSFORMER BLOCK ####
--> The Transformer Block combines Multi-Head Self-Attention and a Feed Forward Network to process token representations.
--> first normalizes the input before it enters the attention layer.
--> then allows each token to gather information from other relevant tokens.
--> after that adds the original input back using a residual connection, helping preserve information.
--> then again normalizes the updated representation before the Feed Forward Network.
--> then applies a non-linear transformation to each token's representation.
--> and finally The second residual connection adds the original representation again, producing the final output of the Transformer Block

In [5]:
class Block(nn.Module):  # here we create Transformer Block combines Multi-Head Self-Attention and a Feed Forward Network to process token representations. 
    def __init__(self,embedding_dim,block_size,n_heads):
        super().__init__()
        self.sa = MultiHeadAttention(embedding_dim,block_size,n_heads) # here we create Multihead attention layer to let token's communicate each other 
        self.ffn = FNN(embedding_dim)   # create Feed Forward network which to transpose token representation Non-linearly
        self.ln1 = nn.LayerNorm(embedding_dim)  # create Layernorm layer which stabilize the value before attention and FFN layer
        self.ln2 = nn.LayerNorm(embedding_dim)  
    def forward(self,x):   
        x = x + self.sa(self.ln1(x)) # self.sa() allows each token to gather information from other relevant tokens.
                                     # x + .... adds the original input back using a residual connection,helping preserve information 
        x = x + self.ffn(self.ln2(x)) #  same here the second residual connection adds the original representation again,producing the fnial output of the transformer Block
        return x

#### MODEL TRAINING ####

In [6]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

In [7]:
print(torch.__version__)
print(torch.cuda.is_available())

2.13.0+cpu
False


In [8]:
## Now we done with liabrary now let's create simple Dataset which is input dataset which we will feed to model
corpus = [
       "hello friends how are you",
       "the tea is very hot",
       "my name is Abhijeet",
       "the roads of delhi are busy",
       "it is raining in mumbai",
       "the train is late again",
       "i love eating samosas and drining tea",
       "holi is my favorite festival",
       "diwali brings lights and sweets",
       "india won the cricket match"

]

In [9]:
## above is the small dataset for our model which we will feed to our model
## now we need to convert above data into a algoritham understandable format like we have to put <end> and <start> in every sentence and then combine whole
## corpus into 1 big paragraph

new_corpus = ["<start> " + s + " <end>" for s in corpus]
final_corpus = " ".join(new_corpus)
print(final_corpus)


<start> hello friends how are you <end> <start> the tea is very hot <end> <start> my name is Abhijeet <end> <start> the roads of delhi are busy <end> <start> it is raining in mumbai <end> <start> the train is late again <end> <start> i love eating samosas and drining tea <end> <start> holi is my favorite festival <end> <start> diwali brings lights and sweets <end> <start> india won the cricket match <end>


In [39]:
final_corpus

'<start> hello friends how are you <end> <start> the tea is very hot <end> <start> my name is Abhijeet <end> <start> the roads of delhi are busy <end> <start> it is raining in mumbai <end> <start> the train is late again <end> <start> i love eating samosas and drining tea <end> <start> holi is my favorite festival <end> <start> diwali brings lights and sweets <end> <start> india won the cricket match <end>'

In [10]:
## now next step is to tokenizer where we going to divide our whole paragraph into token(word) 
chunks = final_corpus.split()  ## here we get list of all words separated by comma


In [11]:
chunks

['<start>',
 'hello',
 'friends',
 'how',
 'are',
 'you',
 '<end>',
 '<start>',
 'the',
 'tea',
 'is',
 'very',
 'hot',
 '<end>',
 '<start>',
 'my',
 'name',
 'is',
 'Abhijeet',
 '<end>',
 '<start>',
 'the',
 'roads',
 'of',
 'delhi',
 'are',
 'busy',
 '<end>',
 '<start>',
 'it',
 'is',
 'raining',
 'in',
 'mumbai',
 '<end>',
 '<start>',
 'the',
 'train',
 'is',
 'late',
 'again',
 '<end>',
 '<start>',
 'i',
 'love',
 'eating',
 'samosas',
 'and',
 'drining',
 'tea',
 '<end>',
 '<start>',
 'holi',
 'is',
 'my',
 'favorite',
 'festival',
 '<end>',
 '<start>',
 'diwali',
 'brings',
 'lights',
 'and',
 'sweets',
 '<end>',
 '<start>',
 'india',
 'won',
 'the',
 'cricket',
 'match',
 '<end>']

In [49]:
## now let's make vocabulary from chunks
## VOCABULARY is nothing but the number of unique words we have in list
vocab = list(set(chunks))
vocab_size = len(vocab)


In [13]:
### EMBEDDING  ##

## now still this words are in form of string and model don't understand string it only understand numbers

embedding = {w:i for i,w in enumerate(vocab)}

In [14]:
embedding

{'how': 0,
 'are': 1,
 'hot': 2,
 'name': 3,
 '<end>': 4,
 'very': 5,
 'drining': 6,
 'hello': 7,
 'Abhijeet': 8,
 'the': 9,
 'i': 10,
 'india': 11,
 'won': 12,
 'roads': 13,
 'love': 14,
 'eating': 15,
 'tea': 16,
 '<start>': 17,
 'brings': 18,
 'raining': 19,
 'holi': 20,
 'my': 21,
 'busy': 22,
 'late': 23,
 'you': 24,
 'of': 25,
 'delhi': 26,
 'favorite': 27,
 'cricket': 28,
 'is': 29,
 'again': 30,
 'train': 31,
 'in': 32,
 'diwali': 33,
 'festival': 34,
 'mumbai': 35,
 'and': 36,
 'match': 37,
 'samosas': 38,
 'sweets': 39,
 'lights': 40,
 'it': 41,
 'friends': 42}

In [15]:
## now above we see we give unique id to each tokens
## now we want only the id's not the name so we can now get only id's
text_id_data = torch.tensor([embedding[c] for c in chunks],dtype=torch.long)

In [16]:
final_corpus

'<start> hello friends how are you <end> <start> the tea is very hot <end> <start> my name is Abhijeet <end> <start> the roads of delhi are busy <end> <start> it is raining in mumbai <end> <start> the train is late again <end> <start> i love eating samosas and drining tea <end> <start> holi is my favorite festival <end> <start> diwali brings lights and sweets <end> <start> india won the cricket match <end>'

In [17]:
text_id_data

tensor([17,  7, 42,  0,  1, 24,  4, 17,  9, 16, 29,  5,  2,  4, 17, 21,  3, 29,
         8,  4, 17,  9, 13, 25, 26,  1, 22,  4, 17, 41, 29, 19, 32, 35,  4, 17,
         9, 31, 29, 23, 30,  4, 17, 10, 14, 15, 38, 36,  6, 16,  4, 17, 20, 29,
        21, 27, 34,  4, 17, 33, 18, 40, 36, 39,  4, 17, 11, 12,  9, 28, 37,  4])

In [18]:
len(text_id_data)

72

In [19]:
## now let's define some hyperparameter of transformer model
context_window = 6  # context window means number of words model take as input to predict next word word
embedding_dim = 32  # number of diamention used to represnt each word in our case 32 [0.1.0.4,-0.9.....]
n_heads = 2  # numbner of attention heads using in transformer
n_layers = 2  # number of encode or decoder layers in transformer like 2 means 2 encode layer 2 decoder layers
lerning_rate = 0.01  # learning rate of mode used in optimizer
epoch = 1500  #number of epoches

In [20]:
text_id_data

tensor([17,  7, 42,  0,  1, 24,  4, 17,  9, 16, 29,  5,  2,  4, 17, 21,  3, 29,
         8,  4, 17,  9, 13, 25, 26,  1, 22,  4, 17, 41, 29, 19, 32, 35,  4, 17,
         9, 31, 29, 23, 30,  4, 17, 10, 14, 15, 38, 36,  6, 16,  4, 17, 20, 29,
        21, 27, 34,  4, 17, 33, 18, 40, 36, 39,  4, 17, 11, 12,  9, 28, 37,  4])

In [23]:
embedding

{'hot': 0,
 'roads': 1,
 'and': 2,
 'holi': 3,
 'india': 4,
 'lights': 5,
 'my': 6,
 'train': 7,
 'delhi': 8,
 'diwali': 9,
 'it': 10,
 'late': 11,
 'cricket': 12,
 'very': 13,
 'mumbai': 14,
 'brings': 15,
 'busy': 16,
 'festival': 17,
 'love': 18,
 'drining': 19,
 'the': 20,
 'Abhijeet': 21,
 'of': 22,
 'in': 23,
 'again': 24,
 'you': 25,
 'favorite': 26,
 'is': 27,
 'i': 28,
 'are': 29,
 '<end>': 30,
 '<start>': 31,
 'how': 32,
 'tea': 33,
 'raining': 34,
 'eating': 35,
 'samosas': 36,
 'match': 37,
 'name': 38,
 'hello': 39,
 'won': 40,
 'sweets': 41,
 'friends': 42}

In [22]:
corpus

['hello friends how are you',
 'the tea is very hot',
 'my name is Abhijeet',
 'the roads of delhi are busy',
 'it is raining in mumbai',
 'the train is late again',
 'i love eating samosas and drining tea',
 'holi is my favorite festival',
 'diwali brings lights and sweets',
 'india won the cricket match']

In [24]:
s = [embedding[i] for i in corpus[0].split()]
s

[7, 42, 0, 1, 24]

In [25]:
# Now let's create function which create batch of input to feed model
def get_batch(batch_size=16): # here we feeding 16 sentences in each batch to model
    # now we have our input data in form of id's let's randomly pick 16 id's from data 
    index = torch.randint(len(text_id_data) - context_window,(batch_size,)) # here we get 16 id's(word) from our id_data list from 0 to 62-6=56 (0-56)
    # Note:- here we do len(text_id_data) - context_window  --> means we avoid to take out of vocabulary word,here we prevent out of vocubulary error
    # now next step is to create input and output for model where input is 6 lenght long sequence and output is 7th word in sequence
    # example if we pick id 7 then input = [7,8,9,10,11,12] and output [8,9,10,11,12,13] 13 is output  if we pick 45 then input is [45,46,47,48,49,50] and output is [46,47,48,49,50,51] etc
    x = torch.stack([text_id_data[i:i+context_window] for i in index]) # here we create 16 tensors each contain 6 id's
    # torch.stack concat 2 ventors into 1 using dimeantion like dim=0--> row wise dim=1 --> column wise
    y = torch.stack([text_id_data[i+1:i+context_window+1] for i in index])  # here we create actual output
    return x,y

    

In [2]:
class TinyGPT(nn.Module):  # TinyGPT class defines the main GPT model structure—basically, how tokens move through embeddings → Transformer blocks → final prediction.
    def __init__(self): 
        super().__init__()
        self.token_encoding = nn.Embedding(vocab_size,embedding_dim) # here we Converts each token ID into a 32-dimensional vector. example hello" → ID 5 → [0.21, -0.14, ..., 0.37]   # 32 numbers
        # If vocab_size = 50, this layer contains a 50 × 32 embedding table.
        self.positional_encoding = nn.Embedding(context_window,embedding_dim) # here we Tells the model where each token is located in the sequence. Each position gets its own 32-dimensional vector.
        self.attentionblock = nn.Sequential(*[Block(embedding_dim,context_window,n_heads) for _ in range(n_layers)]) # Creates 2 Transformer Blocks because:
        # cause n_layers = 2 Each Block contains Multi-Head Attention + Feed Forward Network.
        self.ln = nn.LayerNorm(embedding_dim) # Normalizes the final 32-dimensional representation before prediction.
        self.head = nn.Linear(embedding_dim,vocab_size) # Converts the 32-dimensional representation into one score for every word in the vocabulary.
        # Those 50 numbers are called logits and represent how likely each vocabulary token is to be the next token.

    ## Now let's design The forward() function who defines how input tokens pass through the TinyGPT model during training or prediction
    def forward(self,idx,target=None):
        B,T,C = idx.shape
        tok_enb = self.token_encoding(idx)
        pos_enb = self.positional_encoding(torch.arange(T))
        x = tok_enb + pos_enb
        x = self.attentionblock(x)
        x = self.ln(x)
        logits = self.head(x)
        loss = None
        if target is not None:
            B,T,C = logits.shapes
            self.loss = F.cross_entropy(logits.view(B*T,C),target.view(B*T))
        return logits,loss